<a href="https://colab.research.google.com/github/lendenis2671-boop/sort_number/blob/main/sorty.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [74]:
from google.colab import files
import pandas as pd
import re


In [75]:
uploaded = files.upload()
print(list(uploaded.keys()))

Saving fio_file.xlsx to fio_file (7).xlsx
Saving msg_file.xlsx to msg_file (7).xlsx
Saving phone_file.xlsx to phone_file (8).xlsx
['fio_file (7).xlsx', 'msg_file (7).xlsx', 'phone_file (8).xlsx']


In [76]:
df_msg = pd.read_excel('msg_file.xlsx')
df_fio = pd.read_excel('fio_file.xlsx')
df_phone = pd.read_excel('phone_file.xlsx')

print('msg:', df_msg.shape)
print('fio:', df_fio.shape)
print('phone:', df_phone.shape)


msg: (202, 1)
fio: (162, 4)
phone: (162, 2)


In [77]:
import re

vowels = "аеёиоуыэюяАЕЁИОУЫЭЮЯ"
consonants = "бвгджзйклмнпрстфхцчшщБВГДЖЗЙКЛМНПРСТФХЦЧШЩ"
punct = ".,!?;:-()[]{}\"'«»"

def count_vowels(s):
    return sum(1 for char in str(s) if char in vowels)

def count_consonants(s):
    return sum(1 for char in str(s) if char in consonants)

def count_punct(s):
    return sum(1 for char in str(s) if char in punct)

def only_digits(s):
    res = ""
    for char in str(s):
        if char.isdigit():
            res += char
    return res

def to_intl_ru(phone):
    d = only_digits(phone)
    if len(d) == 10:
        return "+7 (" + d[0:3] + ") " + d[3:6] + "-" + d[6:8] + "-" + d[8:10]
    if len(d) == 11 and (d[0] == '7' or d[0] == '8'):
        return "+7 (" + d[1:4] + ") " + d[4:7] + "-" + d[7:9] + "-" + d[9:11]
    return ""

def is_phone_ru(d):
    return (len(d) == 10) or (len(d) == 11 and d[0] in ['7', '8'])

def extract_phone_from_text(s):
    s = str(s)

    cands = re.findall(r'[\+\d][\d\s\-\(\)]{9,15}', s)
    for c in cands:
        d = only_digits(c)
        if is_phone_ru(d):
            return d
    return ""

In [78]:
text_col = df_msg.columns[0]
msg = df_msg.copy()
msg['Исходная строка'] = msg[text_col].astype(str)
msg['Кол-во символов'] = msg['Исходная строка'].apply(len)
msg['Знаков препинания'] = msg['Исходная строка'].apply(count_punct)
msg['Гласных'] = msg['Исходная строка'].apply(count_vowels)
msg['Согласных'] = msg['Исходная строка'].apply(count_consonants)
msg['номер телефона'] = msg['Исходная строка'].apply(extract_phone_from_text)
msg['корректный номер'] = msg['номер телефона'].apply(is_phone_ru)
msg['номер в международном формате'] = msg['номер телефона'].apply(to_intl_ru)
print('Количество строк в файле:', len(msg))
msg_result = msg[['Исходная строка', 'номер телефона', 'номер в международном формате', 'корректный номер']]
msg_result = msg_result[msg_result['корректный номер'] == True]
msg_result = msg_result[['Исходная строка', 'номер телефона', 'номер в международном формате']]
display(msg_result.head(10))

Количество строк в файле: 202


,Исходная строка,номер телефона,номер в международном формате
0,ЮлияБелова: прошу удалить из базы 894669 Берсе...,79049898850,+7 (904) 989-88-50
1,Прошу удалить из базы неработающих номер +7904...,79043470876,+7 (904) 347-08-76
2,"МАГОМЕДТАЛИЕВА МАРЗИЯТ (анкета 6 649 389) , да...",89885778700,+7 (988) 577-87-00
3,Прошу удалит номер 7 904 894-80-38 Немеляйнен ...,79048948038,+7 (904) 894-80-38
4,Прошу удалить номер из базы неработающих 7 993...,79939499086,+7 (993) 949-90-86
5,Прошу удалить из базы Гущенец Ирина Васильевна...,79509037947,+7 (950) 903-79-47
6,Прошу удалить номер 7 963 579-85-48 Мельникова...,79635798548,+7 (963) 579-85-48
7,прошу удалить номер 77774699969 Мендыгереева А...,77774699969,+7 (777) 469-99-69
8,прошу удалить номер 79000440596 Абросимова Ант...,79000440596,+7 (900) 044-05-96
9,Прошу удалить номер 79008683739 Муравлянцева Н...,79008683739,+7 (900) 868-37-39


In [79]:
msg_result.to_csv('result_messages.csv', index=False, encoding='utf-8-sig')
files.download('result_messages.csv')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [80]:
fio = df_fio.copy()
phone = df_phone.copy()
fio = fio.dropna()
phone = phone.dropna()
phone['цифры'] = phone['Номер телефона'].apply(only_digits)
phone['корректный'] = phone['цифры'].apply(is_phone_ru)
phone['номер в международном формате'] = phone['Номер телефона'].apply(to_intl_ru)
def get_gender(fio_text):
    fio_text = str(fio_text).lower()
    if 'вич' in fio_text:
        return 'М'
    if 'вна' in fio_text:
        return 'Ж'
    return 'Неизвестно'
fio['пол'] = fio['ФИО'].apply(get_gender)
def get_age(date):
    try:
        year = pd.to_datetime(date).year
        return 2024 - year
    except:
        return 0
fio['возраст'] = fio['Дата рождения'].apply(get_age)
final_book = pd.merge(fio, phone, on='ID')
final_book = final_book[final_book['пол'] != 'Неизвестно']
final_book = final_book[final_book['корректный'] == True]
final_book = final_book[['ID', 'ФИО', 'Номер телефона', 'номер в международном формате', 'пол', 'возраст']]
final_book = final_book.sort_values('ID')
display(final_book.head(10))

,ID,ФИО,Номер телефона,номер в международном формате,пол,возраст
23,56398,Колобов Юрий Павлович,79087688950,+7 (908) 768-89-50,М,34
64,56452,Шимко Антоний Сергеевич и Юлия Валерьевна,79056008585,+7 (905) 600-85-85,М,59
2,56461,Бакуленко Татьяна Геннадьевна,79833900843,+7 (983) 390-08-43,Ж,54
1,56461,Бакуленко Татьяна Геннадьевна,79906533398,+7 (990) 653-33-98,Ж,54
42,56500,Никифорова Фаина Вячеславовна,79039088993,+7 (903) 908-89-93,Ж,34
43,56562,Никифорова Фаина Вячеславовна,79939763075,+7 (993) 976-30-75,Ж,54
65,56595,Шимко Антоний Сергеевич и Юлия Валерьевна,79948657503,+7 (994) 865-75-03,М,54
33,56609,Кушнир Марианна Николаевна и Черницов Андрей А...,79065985998,+7 (906) 598-59-98,М,54
53,56610,Стрельникова Анна Анатольевна,79039948847,+7 (903) 994-88-47,Ж,34
18,56629,Гронская Мария Владимировна,79995865868,+7 (999) 586-58-68,Ж,54


In [81]:
final_book.to_csv('result_phone_book.csv', index=False, encoding='utf-8-sig')
files.download('result_phone_book.csv')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [82]:
stat_data = final_book[final_book['пол'] != 'Неизвестно']
stat = stat_data.groupby(['возраст', 'пол']).size().reset_index()
stat.columns = ['возраст', 'пол', 'количество']
stat = stat.sort_values(['возраст', 'пол'])
display(stat)

,возраст,пол,количество
0,34,Ж,10
1,34,М,4
2,36,Ж,4
3,36,М,1
4,54,Ж,27
5,54,М,8
6,59,Ж,6
7,59,М,2


In [83]:
stat.to_csv('result_stats_age_gender.csv', index=False, encoding='utf-8-sig')
files.download('result_stats_age_gender.csv')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>